# 大赛 Baseline（规范版）

本 baseline 仅保留最小可用能力，便于选手二次开发。

要求与约束：
1. 题目输入字段保持不变：question_id, type, difficulty, question，以及可选字段 image
2. 输出字段保持不变：question_id, reasoning_process, answer
3. 不使用脚本入口，改为可实例化类 + 单一解题方法
4. baseline 包含两个核心交付：baseline_agent.py 与 submission.json
5. image 字段仅在含图题出现，内容为图片相对路径；本 baseline 仅说明该字段，不实现图像处理

In [ ]:
# baseline_agent.py
from __future__ import annotations

from dataclasses import dataclass
import ast
import operator
import re
from typing import Any, Dict, List


@dataclass
class BaselineConfig:
    max_reasoning_chars: int = 1200


class BaselineAgent:
    _allowed_ops = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.Mod: operator.mod,
    }

    _allowed_unary_ops = {
        ast.UAdd: operator.pos,
        ast.USub: operator.neg,
    }

    def __init__(self, config: BaselineConfig | None = None) -> None:
        self.config = config or BaselineConfig()

    def solve(self, item: Dict[str, Any]) -> Dict[str, str]:
        q_id = str(item.get("question_id", ""))
        q_type = str(item.get("type", ""))
        difficulty = str(item.get("difficulty", ""))
        question = str(item.get("question", "")).strip()
        image_path = item.get("image")

        reasoning_parts: List[str] = []
        reasoning_parts.append(f"题目类型: {q_type or '未知'}")
        reasoning_parts.append(f"难度: {difficulty or '未知'}")
        if image_path:
            reasoning_parts.append("检测到 image 字段（含图题）。baseline 仅记录路径，不处理图像内容。")

        answer = "请根据题意作答"
        expr = self._extract_math_expression(question)

        if expr:
            try:
                value = self._safe_eval(expr)
                answer = str(value)
                reasoning_parts.append(f"识别到算式: {expr}")
                reasoning_parts.append("使用安全表达式求值器完成计算。")
                reasoning_parts.append(f"计算结果: {answer}")
            except Exception as exc:
                reasoning_parts.append(f"识别到算式: {expr}")
                reasoning_parts.append(f"计算失败，原因: {exc}")
                reasoning_parts.append("已退回基础回答模板。")
        else:
            reasoning_parts.append("未识别到可直接计算的标准算式。")
            reasoning_parts.append("该 baseline 不做复杂推理，等待选手扩展。")

        reasoning = "\n".join(reasoning_parts)
        if len(reasoning) > self.config.max_reasoning_chars:
            reasoning = reasoning[: self.config.max_reasoning_chars].rstrip() + "..."

        return {
            "question_id": q_id,
            "reasoning_process": reasoning,
            "answer": answer,
        }

    def _extract_math_expression(self, text: str) -> str | None:
        normalized = text.replace("×", "*").replace("÷", "/").replace("（", "(").replace("）", ")")
        candidates = re.findall(r"[\d\.\s\+\-\*/\(\)\%\^]+", normalized)
        for candidate in candidates:
            expr = candidate.strip()
            if not expr:
                continue
            if re.search(r"\d", expr) and re.search(r"[\+\-\*/\^%]", expr):
                return expr.replace("^", "**")
        return None

    def _safe_eval(self, expr: str) -> float:
        node = ast.parse(expr, mode="eval")
        value = self._eval_ast(node.body)
        return float(value)

    def _eval_ast(self, node: ast.AST) -> float:
        if isinstance(node, ast.Constant):
            if isinstance(node.value, (int, float)):
                return float(node.value)
            raise ValueError("表达式包含非法常量")

        if isinstance(node, ast.BinOp):
            op_type = type(node.op)
            if op_type not in self._allowed_ops:
                raise ValueError("表达式包含不支持的二元运算")
            left = self._eval_ast(node.left)
            right = self._eval_ast(node.right)
            return float(self._allowed_ops[op_type](left, right))

        if isinstance(node, ast.UnaryOp):
            op_type = type(node.op)
            if op_type not in self._allowed_unary_ops:
                raise ValueError("表达式包含不支持的一元运算")
            value = self._eval_ast(node.operand)
            return float(self._allowed_unary_ops[op_type](value))

        raise ValueError("表达式语法不受支持")

## submission.json 示例内容

{
  "Team_name": "xxx",
  "python_version": "3.12",
  "entry_class": "BaselineAgent",
  "entry_method": "solve"
}

说明：
- 主办方根据 entry_class 实例化类，并调用 entry_method 进行解题。
- 本 baseline 只保留一个解题方法 solve。

## 接口示例（保持不变）

输入（单题）示例：
{
  "question_id": "Q1",
  "type": "计算题",
  "difficulty": "简单",
  "question": "计算 2*(3+4)",
  "image": "images/q1.png"  // 可选，仅含图题提供
}

输出（单题）示例：
{
  "question_id": "Q1",
  "reasoning_process": "...",
  "answer": "14.0"
}

说明：
- 评测系统可直接实例化 BaselineAgent 并调用 solve。
- 选手后续可以只替换 BaselineAgent 的内部策略，不改输入输出字段。